# Demo 3 — Run a validated cleaning pipeline

**Learning objectives**

- Run one restartable raw → audit → decide → transform → validate → save pipeline.
- Express the clean-data contract as executable invariants.
- Write and read back a clean dataset, issue audit, and decision log without relying on stored notebook state.

Colab is the default launch experience; local Jupyter runs the same cells. See `DEMO_GUIDE.md` for launch, output, and destructive-fixture rehearsal instructions. GitHub source opened in Colab is not automatically updated by edits in the Colab tab.

Compatibility candidate: Python 3.12.13, NumPy 2.0.2, pandas 3.0.3. This is not the final course lock until fresh local and Colab certification is complete. The fixture contains invented teaching records only.

In [ ]:
from importlib.metadata import PackageNotFoundError, version
import subprocess
import sys

PANDAS_CANDIDATE = "3.0.3"
try:
    installed_pandas = version("pandas")
except PackageNotFoundError:
    installed_pandas = None
if installed_pandas != PANDAS_CANDIDATE:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", f"pandas=={PANDAS_CANDIDATE}"],
        check=True,
    )

import numpy as np
import pandas as pd

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)

## Acquire, verify, and preserve the raw artifact

The portable bootstrap uses the committed fixture when the repository is present and otherwise writes the same supplied bytes into runtime-local storage. It creates the output directory from visible code and verifies the raw checksum before parsing.

In [ ]:
from hashlib import sha256
from pathlib import Path

SOURCE_NAME = "supplied_people_raw.csv"
SOURCE_RELATIVE_PATH = Path("05") / "demo" / "data" / SOURCE_NAME
EXPECTED_SHA256 = "7b3223154756aa59f2f00027ddbadaa225eeee51ad75d0df91de1fd8d14abe2d"
SUPPLIED_SOURCE_BYTES = (
    b"record_id,full_name,site,status,age_text,visit_date\n"
    b"R001, Alice Smith , north,Active,34,2026-01-15\n"
    b"R002,BOB JONES,North,active,unknown,2026-02-30\n"
    b"R002,BOB JONES,North,active,unknown,2026-02-30\n"
    b"R003, Carla Ruiz ,SOUTH,pending,-9,2026-03-01\n"
    b"R004,,south,NA,45,\n"
    b"R005,Evan Li,west,complete,52,2026-02-14\n"
)


def find_course_file(start, relative_path):
    current = start.resolve()
    while True:
        candidate = current / relative_path
        if candidate.is_file():
            return candidate
        if current.parent == current:
            return None
        current = current.parent


DATA_PATH = find_course_file(Path.cwd(), SOURCE_RELATIVE_PATH)
lecture_readme = find_course_file(Path.cwd(), Path("05") / "README.md")
if DATA_PATH is None:
    data_dir = Path.cwd() / "data"
    data_dir.mkdir(parents=True, exist_ok=True)
    DATA_PATH = data_dir / SOURCE_NAME
    DATA_PATH.write_bytes(SUPPLIED_SOURCE_BYTES)

actual_sha256 = sha256(DATA_PATH.read_bytes()).hexdigest()
assert actual_sha256 == EXPECTED_SHA256, "Unexpected raw fixture content"
demo_base = Path.cwd() if lecture_readme is None else lecture_readme.parent / "demo"
OUTPUT_DIR = demo_base / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

raw = pd.read_csv(DATA_PATH, keep_default_na=False)
raw_snapshot = raw.copy(deep=True)
print("Input:", DATA_PATH)
print("Output directory:", OUTPUT_DIR)
print("Raw shape:", raw.shape)

## Audit without mutation

The audit function returns affected-row or affected-token counts with explicit labels. Conversion calls here are probes: they detect failures but do not make cleaning decisions.

In [ ]:
EXPECTED_COLUMNS = ["record_id", "full_name", "site", "status", "age_text", "visit_date"]
EXACT_DATE_PATTERN = r"[0-9]{4}-[0-9]{2}-[0-9]{2}"


def audit_person_records(raw_table):
    schema_matches = list(raw_table.columns) == EXPECTED_COLUMNS
    age_sentinel = raw_table["age_text"].isin(["unknown", "-9"])
    status_sentinel = raw_table["status"].eq("NA")
    age_source = raw_table["age_text"].replace({"unknown": pd.NA, "-9": pd.NA})
    age_probe = pd.to_numeric(age_source, errors="coerce")
    age_parse_failure = age_probe.isna() & age_source.notna()
    age_finite = age_probe.notna() & age_probe.abs().lt(float("inf"))
    age_noninteger = age_finite & age_probe.mod(1).ne(0)
    age_integer = age_finite & ~age_noninteger
    age_range_failure = age_integer & ~age_probe.between(0, 120)
    date_source = raw_table["visit_date"].replace({"": pd.NA})
    date_text_ok = date_source.str.fullmatch(EXACT_DATE_PATTERN, na=False)
    date_format_failure = date_source.notna() & ~date_text_ok
    date_probe = pd.to_datetime(
        date_source.where(date_text_ok, pd.NA),
        format="%Y-%m-%d",
        errors="coerce",
    )
    date_failure = date_format_failure | (date_text_ok & date_probe.isna())
    normalized_site = raw_table["site"].str.strip().str.lower()
    normalized_status = raw_table["status"].str.strip().str.lower()

    return pd.DataFrame([
        {"issue": "schema mismatch", "count": int(not schema_matches)},
        {"issue": "empty full-name tokens", "count": int(raw_table["full_name"].eq("").sum())},
        {"issue": "empty date tokens", "count": int(raw_table["visit_date"].eq("").sum())},
        {"issue": "age sentinel tokens", "count": int(age_sentinel.sum())},
        {"issue": "status sentinel tokens", "count": int(status_sentinel.sum())},
        {"issue": "age parse failures", "count": int(age_parse_failure.sum())},
        {"issue": "numeric but noninteger age values", "count": int(age_noninteger.sum())},
        {"issue": "age values outside 0 through 120", "count": int(age_range_failure.sum())},
        {"issue": "date parse failures", "count": int(date_failure.sum())},
        {"issue": "rows in exact duplicate sets", "count": int(raw_table.duplicated(keep=False).sum())},
        {"issue": "rows with repeated candidate IDs", "count": int(raw_table.duplicated(subset=["record_id"], keep=False).sum())},
        {"issue": "site values needing format normalization", "count": int(raw_table["site"].ne(normalized_site).sum())},
        {"issue": "status values needing format normalization", "count": int((raw_table["status"].ne(normalized_status) & ~status_sentinel).sum())},
        {"issue": "unexpected site values", "count": int((~normalized_site.isin(["north", "south", "west"])).sum())},
        {"issue": "unexpected non-sentinel status values", "count": int((~normalized_status.isin(["active", "pending", "complete", "na"])).sum())},
    ])


issue_audit = audit_person_records(raw)
issue_audit

## Record decisions

The decision table carries the human reasoning that assertions cannot supply. Every later transformation must trace to one of these rows.

In [ ]:
decision_table = pd.DataFrame([
    {"field": "full_name", "issue": "empty token", "action": "convert to missing and retain row", "reason": "name is optional in this de-identified exercise"},
    {"field": "status", "issue": "NA sentinel", "action": "convert to missing and retain row", "reason": "the source dictionary defines NA as unknown status"},
    {"field": "age_text", "issue": "unknown and -9 sentinels", "action": "convert to missing; do not impute", "reason": "no defensible person-level age estimate is supplied"},
    {"field": "age_text", "issue": "nonnumeric or numeric-but-noninteger value", "action": "convert to missing and flag; do not round", "reason": "the integer-age schema supplies no correction"},
    {"field": "visit_date", "issue": "empty or invalid calendar date", "action": "convert to missing and flag", "reason": "inventing a date would change record meaning"},
    {"field": "all columns", "issue": "one exact repeated submission", "action": "retain first exact row", "reason": "the repeated rows carry identical information"},
])
decision_table

## Transform a working copy

The function applies only documented column operations. It derives the exact-duplicate keep mask before normalization so raw-distinct records are never collapsed merely because cleaning makes their values look alike.

In [ ]:
def clean_person_records(raw_table):
    exact_duplicate_keep_mask = ~raw_table.duplicated(keep="first")
    result = raw_table.copy(deep=True)

    result["full_name"] = result["full_name"].replace({"": pd.NA})
    result["status"] = result["status"].replace({"NA": pd.NA})
    result["age_text"] = result["age_text"].replace({"unknown": pd.NA, "-9": pd.NA})
    result["visit_date"] = result["visit_date"].replace({"": pd.NA})

    result["full_name"] = result["full_name"].str.strip().str.title()
    result["site"] = result["site"].str.strip().str.lower()
    result["status"] = result["status"].str.strip().str.lower()

    age_numeric = pd.to_numeric(result["age_text"], errors="coerce")
    age_finite = age_numeric.notna() & age_numeric.abs().lt(float("inf"))
    age_integer = age_finite & age_numeric.mod(1).eq(0)
    result["age_text"] = age_numeric.where(age_integer, pd.NA).astype("Int64")
    result = result.rename(columns={"age_text": "age"})

    date_text_ok = result["visit_date"].str.fullmatch(EXACT_DATE_PATTERN, na=False)
    result["visit_date"] = pd.to_datetime(
        result["visit_date"].where(date_text_ok, pd.NA),
        format="%Y-%m-%d",
        errors="coerce",
    )

    result = result.loc[exact_duplicate_keep_mask].copy()
    result["needs_review"] = result["age"].isna() | result["visit_date"].isna()
    return result


cleaned = clean_person_records(raw)
cleaned

## Validate explicit invariants

A **validation invariant** is a condition that must be true after the stage. Assertions stop the pipeline when a contract is broken; they do not prove that the recorded decisions were wise.

In [ ]:
validation_results = pd.Series({
    "raw preserved": raw.equals(raw_snapshot),
    "expected rows after one exact duplicate removal": len(cleaned) == 5,
    "record ID present": cleaned["record_id"].notna().all(),
    "record ID unique": cleaned["record_id"].is_unique,
    "site allowed": cleaned["site"].isin(["north", "south", "west"]).all(),
    "status allowed when present": cleaned["status"].dropna().isin(["active", "pending", "complete"]).all(),
    "age has nullable integer dtype": str(cleaned["age"].dtype) == "Int64",
    "age in range when present": cleaned["age"].dropna().between(0, 120).all(),
    "visit date has datetime dtype": pd.api.types.is_datetime64_any_dtype(cleaned["visit_date"].dtype),
}, name="passed")

assert validation_results.all(), validation_results[~validation_results]
validation_results

## Check two easy-to-miss rules

The first regression keeps raw-distinct rows even when normalization makes their cleaned values equal. The second converts fractional numeric age text to missing without rounding and sends it to review.

In [ ]:
normalization_distinct_raw = pd.DataFrame({
    "record_id": ["R100", "R100"],
    "full_name": [" Alice Example ", "alice example"],
    "site": ["North", "north"],
    "status": ["Active", "active"],
    "age_text": ["40", "40"],
    "visit_date": ["2026-01-01", "2026-01-01"],
})
assert not normalization_distinct_raw.duplicated(keep=False).any()
assert len(clean_person_records(normalization_distinct_raw)) == 2

fractional_age_raw = pd.DataFrame({
    "record_id": ["R200"],
    "full_name": ["Fractional Age"],
    "site": ["north"],
    "status": ["active"],
    "age_text": ["40.5"],
    "visit_date": ["2026-01-01"],
})
fractional_audit = audit_person_records(fractional_age_raw).set_index("issue")["count"]
fractional_result = clean_person_records(fractional_age_raw)
assert fractional_audit["numeric but noninteger age values"] == 1
assert fractional_result["age"].isna().all()
assert fractional_result["needs_review"].all()
assert str(fractional_result["age"].dtype) == "Int64"
print("Regression checks passed")

## Save the audit trail and verify the round trip

The decision log records provenance and before/after row counts. The final cell replaces all three generated CSVs, reads them back with an explicit clean schema, and compares the clean round trip exactly with the in-memory result.

In [ ]:
decision_log = decision_table.copy()
decision_log["source"] = SOURCE_NAME
decision_log["source_sha256"] = actual_sha256
decision_log["rows_before"] = len(raw)
decision_log["rows_after"] = len(cleaned)

CLEANED_PATH = OUTPUT_DIR / "cleaned_people.csv"
AUDIT_PATH = OUTPUT_DIR / "issue_audit.csv"
DECISION_LOG_PATH = OUTPUT_DIR / "decision_log.csv"
cleaned.to_csv(CLEANED_PATH, index=False)
issue_audit.to_csv(AUDIT_PATH, index=False)
decision_log.to_csv(DECISION_LOG_PATH, index=False)

round_trip = pd.read_csv(
    CLEANED_PATH,
    dtype={
        "record_id": "string",
        "full_name": "string",
        "site": "string",
        "status": "string",
        "age": "Int64",
        "visit_date": "string",
        "needs_review": "boolean",
    },
)
round_trip_date_text_ok = round_trip["visit_date"].str.fullmatch(EXACT_DATE_PATTERN, na=False)
assert (round_trip["visit_date"].isna() | round_trip_date_text_ok).all()
round_trip["visit_date"] = pd.to_datetime(
    round_trip["visit_date"].where(round_trip_date_text_ok, pd.NA),
    format="%Y-%m-%d",
    errors="coerce",
)
expected_round_trip = cleaned.reset_index(drop=True).astype({
    "record_id": "string",
    "full_name": "string",
    "site": "string",
    "status": "string",
    "age": "Int64",
    "needs_review": "boolean",
})
pd.testing.assert_frame_equal(round_trip, expected_round_trip, check_exact=True)

audit_round_trip = pd.read_csv(AUDIT_PATH)
decision_round_trip = pd.read_csv(DECISION_LOG_PATH)
assert audit_round_trip.equals(issue_audit)
assert len(decision_round_trip) == len(decision_log) == 6
assert decision_round_trip["source_sha256"].eq(EXPECTED_SHA256).all()
assert sha256(DATA_PATH.read_bytes()).hexdigest() == EXPECTED_SHA256

print("Demo 3 fresh-run cleaning outputs verified")
print(CLEANED_PATH)
print(AUDIT_PATH)
print(DECISION_LOG_PATH)